# Word Sentiment of Reddit WSB - Latest Data

This notebook analyzes the latest public `r/wallstreetbets` post titles and bodies using current project outputs.

It now:
- refreshes the latest WSB dataset when needed,
- computes VADER sentiment for titles and bodies,
- summarizes positive / neutral / negative message counts,
- visualizes top words and word clouds,
- saves charts under `outputs/latest_wsb_analysis/charts/`.

## Step 1 - Setup

In [1]:
from collections import Counter
from pathlib import Path
import json
import re
import sys
from typing import cast

import matplotlib.pyplot as plt
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud, STOPWORDS

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

if not (ROOT / "src").exists():
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.wsb_latest.pipeline import ensure_nltk_resources, run_pipeline

ensure_nltk_resources()
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets


## Step 2 - Configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
REFRESH_WSB_DATA = True
TOP_WORDS = 20
MIN_WORD_LENGTH = 3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh and load the latest WSB posts

In [3]:
posts_path = OUTPUT_DIR / "latest_wsb_posts.csv"

if REFRESH_WSB_DATA or (not posts_path.exists()):
    result = run_pipeline(top_n=10, per_feed=100, price_period="1y", output_dir=OUTPUT_DIR)
    print(json.dumps(result, indent=2))

posts_df = cast(pd.DataFrame, pd.read_csv(str(posts_path)))
posts_df["created_utc"] = pd.to_datetime(posts_df["created_utc"], utc=True)
posts_df[["created_utc", "feed", "title", "body", "score", "num_comments"]].head(10)

{
  "data_source": "reddit_public_json",
  "posts": 166,
  "tickers": [
    "SPY",
    "MSTR",
    "RKLB",
    "RACE",
    "AMD",
    "ASTS",
    "TSLA",
    "QQQ",
    "NVDA",
    "NFLX"
  ],
  "output_dir": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis"
}


,created_utc,feed,title,body,score,num_comments
0,2026-04-20 02:30:01+00:00,hot,Blue Orgin “accidentally” deploys their compet...,"You can’t make this shit up, out of the hundre...",61,15
1,2026-04-20 02:14:04+00:00,hot,Jensen’s Leather Jacket is the only thing keep...,# How I Learned to Stop Worrying and Love the ...,119,44
2,2026-04-20 01:26:43+00:00,hot,Straight of Hormuz smp prediction,NaN,216,40
3,2026-04-20 01:15:48+00:00,hot,[$MSTR] I have deemed it necessary to take out...,NaN,95,94
4,2026-04-20 00:41:23+00:00,hot,Market hasn’t even opened yet,Rape meeeeeee!!! Rape meeeeeee!!,441,159
5,2026-04-20 00:11:27+00:00,hot,Upcoming Stock Market Drop Will Be Epic Fury,**Fundamental Analysis:**\n\nEconomic backdrop...,1024,463
6,2026-04-19 19:57:25+00:00,hot,"What Are Your Moves Tomorrow, April 20, 2026",This post contains content not supported on ol...,235,5827
7,2026-04-19 19:53:29+00:00,hot,I made my first M on friday.,NaN,506,90
8,2026-04-19 18:40:04+00:00,hot,I’d like to but a large Wendys combo on Monday,NaN,91,67
9,2026-04-19 18:35:00+00:00,hot,CNBC: Trump says talks between U.S. and Iran t...,Just in time before the futures open. -2% day ...,1617,326


## Step 4 - Clean text and score title/body sentiment

In [4]:
analyzer = SentimentIntensityAnalyzer()
stop_words = set(STOPWORDS) | {"https", "http", "www", "com", "amp", "wallstreetbets", "wsb"}


def clean_text(text):
    text = str(text or "").lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return [
        token
        for token in clean_text(text).split()
        if len(token) >= MIN_WORD_LENGTH and token not in stop_words
    ]

posts_df["clean_title"] = posts_df["title"].apply(clean_text)
posts_df["clean_body"] = posts_df["body"].apply(clean_text)
posts_df["title_sentiment"] = posts_df["clean_title"].apply(lambda text: analyzer.polarity_scores(text)["compound"])
posts_df["body_sentiment"] = posts_df["clean_body"].apply(lambda text: analyzer.polarity_scores(text)["compound"])
posts_df["title_tokens"] = posts_df["clean_title"].apply(tokenize)
posts_df["body_tokens"] = posts_df["clean_body"].apply(tokenize)
posts_df[["title", "title_sentiment", "body_sentiment"]].head(10)

,title,title_sentiment,body_sentiment
0,Blue Orgin “accidentally” deploys their compet...,-0.6705,-0.2441
1,Jensen’s Leather Jacket is the only thing keep...,0.0000,0.8154
2,Straight of Hormuz smp prediction,0.2263,0.0000
3,[$MSTR] I have deemed it necessary to take out...,0.0000,0.0000
4,Market hasn’t even opened yet,0.0000,-0.8860
5,Upcoming Stock Market Drop Will Be Epic Fury,-0.7003,-0.9771
6,"What Are Your Moves Tomorrow, April 20, 2026",0.0000,-0.2411
7,I made my first M on friday.,0.0000,0.0000
8,I’d like to but a large Wendys combo on Monday,0.1901,0.0000
9,CNBC: Trump says talks between U.S. and Iran t...,0.0000,-0.2500


## Step 5 - Summarize current title/body sentiment

In [5]:
def label_sentiment(value):
    if value > 0.05:
        return "positive"
    if value < -0.05:
        return "negative"
    return "neutral"

posts_df["title_label"] = posts_df["title_sentiment"].apply(label_sentiment)
posts_df["body_label"] = posts_df["body_sentiment"].apply(label_sentiment)

sentiment_summary_df = pd.DataFrame({
    "title": posts_df["title_label"].value_counts(),
    "body": posts_df["body_label"].value_counts(),
}).fillna(0).astype(int)
sentiment_summary_df

,title,body
neutral,90,76
positive,43,49
negative,33,41


## Step 6 - Visualize top title/body words

In [6]:
title_counts = Counter(token for tokens in posts_df["title_tokens"] for token in tokens)
body_counts = Counter(token for tokens in posts_df["body_tokens"] for token in tokens)

title_top_df = pd.DataFrame(title_counts.most_common(TOP_WORDS), columns=["word", "count"])
body_top_df = pd.DataFrame(body_counts.most_common(TOP_WORDS), columns=["word", "count"])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(title_top_df["word"], title_top_df["count"], color="#1f77b4")
axes[0].set_title("Top title words")
axes[1].barh(body_top_df["word"], body_top_df["count"], color="#d62728")
axes[1].set_title("Top body words")
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
word_bar_path = CHARTS_DIR / "wsb_top_words_latest.png"
plt.savefig(word_bar_path, dpi=180)
plt.show()
print(f"Saved: {word_bar_path}")

Saved: C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\charts\wsb_top_words_latest.png


C:\Users\big_j\AppData\Local\Temp\ipykernel_609740\2286019212.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7 - Generate title and body word clouds

In [7]:
title_cloud = WordCloud(width=1200, height=600, background_color="white", stopwords=stop_words).generate(" ".join(posts_df["clean_title"]))
body_cloud = WordCloud(width=1200, height=600, background_color="white", stopwords=stop_words).generate(" ".join(posts_df["clean_body"]))

fig, axes = plt.subplots(2, 1, figsize=(14, 12))
axes[0].imshow(title_cloud, interpolation="bilinear")
axes[0].set_title("Latest WSB title word cloud")
axes[0].axis("off")
axes[1].imshow(body_cloud, interpolation="bilinear")
axes[1].set_title("Latest WSB body word cloud")
axes[1].axis("off")
plt.tight_layout()
wordcloud_path = CHARTS_DIR / "wsb_wordclouds_latest.png"
plt.savefig(wordcloud_path, dpi=180)
plt.show()
print(f"Saved: {wordcloud_path}")

Saved: C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\charts\wsb_wordclouds_latest.png


C:\Users\big_j\AppData\Local\Temp\ipykernel_609740\2598415362.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8 - Save refreshed sentiment datasets

In [8]:
title_sentiment_path = OUTPUT_DIR / "wsb_title_sentiment_latest.csv"
body_sentiment_path = OUTPUT_DIR / "wsb_body_sentiment_latest.csv"
summary_path = OUTPUT_DIR / "wsb_word_sentiment_summary.csv"

posts_df[["created_utc", "title", "clean_title", "title_sentiment", "title_label"]].to_csv(title_sentiment_path, index=False)
posts_df[["created_utc", "body", "clean_body", "body_sentiment", "body_label"]].to_csv(body_sentiment_path, index=False)
sentiment_summary_df.to_csv(summary_path)

print(title_sentiment_path)
print(body_sentiment_path)
print(summary_path)

C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\wsb_title_sentiment_latest.csv
C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\wsb_body_sentiment_latest.csv
C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\wsb_word_sentiment_summary.csv


## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "posts_analyzed": int(len(posts_df)),
    "title_wordcloud": str(wordcloud_path.resolve()),
    "top_word_chart": str(word_bar_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "posts_analyzed": 166,
  "title_wordcloud": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis\\charts\\wsb_wordclouds_latest.png",
  "top_word_chart": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis\\charts\\wsb_top_words_latest.png",
  "output_dir": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis"
}
